# 00 - Our cluster architecture

This notebook explains the architecture defined in this repository. Read it before the setup walkthrough or use it to introduce the demonstration. It describes what the setup creates, rather than claiming that a cluster is currently running.

The default command-based demonstration starts Minikube, Headlamp, monitoring, four data services, Jupyter, HDFS, Spark and Kafka. YARN/Hive belong to separate sessions; Airflow/Trino/lineage are later lesson manifests.

[Quick setup by command](QUICKSTART.md) | [Detailed setup: Ubuntu preparation](00-ubuntu-workspace.ipynb) | [Kubernetes concepts introduction](../00-kubernetes-introduction/00-kubernetes-introduction.ipynb) | [Operations and browser URLs](../README.md)

## 01 - The complete picture

![Current repository architecture](assets/cluster-overview.png)

Read the picture from the outside inward. An Ubuntu machine hosts the source checkout, browser and Docker Engine. Docker runs one Minikube node named `bigdata`. Inside that node, containerd runs the Kubernetes application containers. The control plane and workloads share this single node.

Most applications run in namespace `bigdata`. Headlamp runs in `headlamp`, and cluster system components run in `kube-system`. Namespaces group resources; they are not separate machines.

Green application boxes are the current demonstration group. Amber boxes represent separate or later lessons. The diagram is a logical view: each box can contain several Pods, not an extra physical machine.

## 02 - Where everything runs

| Layer | What we have | Purpose |
|---|---|---|
| Ubuntu host | Ordinary login, Docker Engine, Minikube/kubectl tools, browser | Runs and manages the local demonstration |
| Source checkout | `~/workspace` by default; another location is supported | Notebooks, manifests, Python/PySpark, SQL and DAG source |
| Docker node container | `bigdata` | The single Minikube node |
| Kubernetes control plane | API server, scheduler, controllers and etcd | Accept desired configuration, assign Pods and maintain cluster state |
| Node runtime | kubelet and containerd, plus node networking | Run and supervise application containers |
| System services | CoreDNS, local storage provisioner, metrics-server | Service discovery, PVC provisioning and current resource usage |
| Application namespaces | `bigdata`, plus `headlamp` for UI | Group workloads and their configuration |

`versions.env` configures four CPUs and 8192 MiB for the node. The host needs additional capacity for Ubuntu and its other applications. This is a local demonstration, with one node and local storage; it has no multi-node failover.

## 03 - The current demonstration components

| Component | High-level role | What to look for |
|---|---|---|
| Headlamp | Browser view of Kubernetes workloads, Services, storage, events and logs | Viewer identity; it does not expose Secret values |
| SeaweedFS | Supplies S3-compatible object storage | Warehouse objects; master, filer, volume and admin pages |
| PostgreSQL | Relational database and Polaris service-state backend | `postgres-0`; database state on its PVC |
| Polaris | REST catalog for Iceberg | Catalog/table registration; depends on PostgreSQL and S3 |
| MySQL | Relational database exercises; later Hive metastore backend | `mysql-0`; database state on its PVC |
| JupyterLab | Interactive Python/PySpark workbench | Opens project source at `/workspace`; uses the shared image |
| HDFS NameNode | Tracks HDFS directories, file names and block locations | HDFS filesystem metadata |
| HDFS DataNode | Stores HDFS file blocks | The single DataNode in this demonstration |
| Spark master | Coordinates this standalone Spark cluster | Registers workers and schedules Spark application resources |
| Spark worker | Runs Spark executors and tasks | Executes distributed work submitted by an application driver |
| Kafka broker | Stores event streams in topics | Producers write records; consumers read them |
| ZooKeeper | Coordinates this repository's ZooKeeper-backed Kafka setup | Starts before Kafka; this repository does not use KRaft |

The shared image is `dataeng-training:1.2.0`. It packages software used by the workbench and compute components. Databases, object storage and monitoring also use their own images pinned in manifests. A Docker image contains software; a PVC contains persistent application data.

## 04 - Data and metadata paths

![Application and metadata relationships](assets/cluster-data-paths.png)

The arrows illustrate selected application/dependency paths. They do not imply that every connection is activated immediately after deployment. Connectors, credentials and actual jobs are supplied by the relevant exercises.

| Example path | Meaning |
|---|---|
| Jupyter driver -> Spark master -> worker | A configured PySpark application requests standalone Spark resources; executors process tasks on the worker |
| Spark -> HDFS | Spark can read/write files through the NameNode; file bytes are held by the DataNode |
| Iceberg client -> Polaris -> PostgreSQL | Polaris serves catalog operations and persists catalog service state in PostgreSQL |
| Iceberg client -> SeaweedFS S3 | The client reads/writes Iceberg data and metadata files in the object warehouse; table bytes are not stored inside Polaris |
| Producer -> Kafka -> consumer | Exercises exchange event records; the broker persists records and ZooKeeper coordinates broker state |
| Hive metastore -> MySQL, with Hive data in HDFS | In the separate Hive session, schema/table metadata and actual dataset files live in different services |

For an Iceberg exercise, catalog registration and object storage are different responsibilities. Polaris can describe where a table is located; the compatible client accesses its files through S3. Likewise, HDFS metadata and HDFS data blocks have different owners.

The NameNode/DataNode pair is a single local demonstration filesystem, not a redundant production HDFS installation. Spark here is standalone; starting Spark does not require the separate YARN lesson.

## 05 - Components reserved for other sessions

| Component | What it adds | Current scope |
|---|---|---|
| YARN ResourceManager / NodeManager | Allocate and execute Hadoop application containers | Separate session on this small node; YARN containers are not Kubernetes Pods |
| MapReduce HistoryServer | Shows completed MapReduce job history | Part of the YARN lesson; long MapReduce test is separate |
| Hive metastore / HiveServer2 | Table metadata service and SQL access to warehouse data | Separate session; schema setup and MySQL required |
| Airflow | Schedules workflows and DAGs | Later lesson; shares source/DAG files and uses PostgreSQL metadata |
| Trino | SQL queries across configured data sources | Later lesson; manifests include relational, Hive and Iceberg connectors |
| Marquez API / web | Stores and displays OpenLineage metadata | Later lesson; instrumentation must emit lineage events |
| Adminer | Optional browser client for PostgreSQL/MySQL | Optional deployment; not part of `setup-demo.sh` |

These services exist in the repository but are not all started together. Use staged sessions on the 8 GiB profile. A manifest being present does not mean its lesson has been executed or validated.

## 06 - Monitoring is part of the architecture

![Monitoring and storage architecture](assets/cluster-observability-storage.png)

| Component | What it collects or displays |
|---|---|
| Alloy | Collects application/Headlamp container logs and configured file-log output, then sends logs to Loki |
| Loki | Stores logs; Grafana queries them |
| Prometheus | Scrapes configured native/JMX application endpoints and kubelet CPU/memory metrics |
| Grafana | Displays provisioned performance/log dashboards using Prometheus and Loki |
| Jaeger | Receives and displays traces emitted by instrumented code through OTLP |
| metrics-server | Provides current resource usage for `kubectl top` and the cluster UI; it is separate from historical Prometheus metrics |

Monitoring starts before the data services so that their first startup logs can be collected. Some compute Pods use a `file-logs` sidecar to forward configured log files to stdout for collection. Metrics targets for inactive lessons can be down until those services start.

Installing Jaeger does not automatically instrument every application. Installing Marquez later also does not automatically emit lineage. Traces and lineage require compatible emitters/instrumentation in their exercises.

## 07 - Source files and persistent data

The chosen host checkout is mounted into the node at `/course-workspace`, then into Jupyter at `/workspace`. A helper keeps the mount and its loopback SSH tunnel alive. Later Airflow shares the checkout and mounts DAGs at `/opt/airflow/dags`. `/workspace` inside a container is a mount location; it does not require another `workspace` folder inside the repository.

Application data uses PVCs and locally provisioned PVs. With the Docker driver, node storage lives in Docker volume `bigdata`. The host source checkout and the node's local data have different lifetimes.

| Data owner | Storage arrangement |
|---|---|
| SeaweedFS | PVC `seaweedfs-data`, mounted at `/data` |
| PostgreSQL | PVC `postgres-data`, mounted at `/var/lib/postgresql/data` |
| MySQL | PVC `mysql-data`, mounted at `/var/lib/mysql` |
| Polaris | Uses PostgreSQL and S3; no independent data PVC |
| HDFS | NameNode metadata and DataNode blocks use separate StatefulSet volume claims |
| Kafka / ZooKeeper | Broker records and coordination data use their configured persistent claims |
| Monitoring | Loki, Prometheus and other stateful services use their declared storage |
| Source notebooks and code | Host checkout, mounted into the workbench; stored outside node volumes |

A Pod restart or scaling to zero normally retains PVC data. `minikube -p bigdata stop` retains the profile and storage. Deleting the profile destroys its local persistent data, while the source checkout and host-built shared image remain. Local persistence is not a backup.

## 08 - How the browser reaches the applications

The browser runs on Ubuntu, including inside the Guacamole desktop. `kubectl port-forward` connects a chosen local port to a running service's Pod. The browser opens the local URL while that forwarding terminal remains open.

For example, NameNode uses `http://127.0.0.1:19870` locally and port `9870` inside the cluster. Jupyter uses local `8889` and internal `8888`. Local ports can differ to avoid conflicts. Internal service names such as `namenode.bigdata.svc.cluster.local` are Kubernetes DNS names and are not normally resolved by the host browser.

Headlamp can also use the Minikube service URL shown in notebook 01. The complete [browser table and forward commands](../README.md#local-browser-uis) include Pod/container names and distinguish actual UIs from API endpoints. PostgreSQL/MySQL and Kafka do not acquire native browser UIs just because they run in Kubernetes.

## 09 - Why this startup order

| Order | Group | Reason |
|---|---|---|
| 1 | Ubuntu tools and ordinary Docker access | The login account must be able to run the node |
| 2 | Minikube, DNS, storage, namespace and example Secrets | Application workloads need a usable cluster foundation |
| 3 | Headlamp, metrics-server and monitoring | View cluster state and capture startup telemetry |
| 4 | S3 -> PostgreSQL -> Polaris -> MySQL | Polaris requires its storage/database dependencies; MySQL supports SQL and later Hive |
| 5 | Shared image import and workspace mount | Host Docker and node containerd have separate image stores; workbench needs code files |
| 6 | Jupyter -> HDFS -> Spark -> ZooKeeper/Kafka | Readiness is checked sequentially before the demonstration is handed over |

`bash setup-demo.sh` follows this deployment order without running exercise tests. `cluster.sh start all` covers only the four initial data services. It is not a command to start every application in the diagram.

## 10 - Inspect this architecture after setup

These commands inspect the already running cluster; they do not create or start it. Paste Bash commands into the Ubuntu terminal, not a Python notebook kernel. Skip them while the environment is intentionally clean/stopped.

```bash
kubectl --context=bigdata get nodes -o wide
kubectl --context=bigdata get namespaces
kubectl --context=bigdata -n bigdata get deployment,statefulset,pods
kubectl --context=bigdata -n bigdata get services
kubectl --context=bigdata -n bigdata get pvc
kubectl --context=bigdata get pv
kubectl --context=bigdata -n headlamp get pods
```

Find the application containers in each Pod, including file-log sidecars:

```bash
kubectl --context=bigdata -n bigdata get pods   -o custom-columns='POD:.metadata.name,CONTAINERS:.spec.containers[*].name'
```

The YAML files in `platform/`, `monitoring/`, `seaweedfs/`, `postgresql/`, `polaris/`, `mysql/`, `jupyter/`, `hadoop/`, `spark/`, `kafka/` and the later component folders are the source of truth. Running resources may differ if you paused a lesson or changed configuration.

## 11 - A simple way to explain the demonstration

Start with the Ubuntu host and its one Minikube node. Show the browser and the source checkout. Then explain three responsibilities: services store data, compute services process it, and monitoring shows what happened.

Follow one example: a PySpark notebook submits work to Spark, the worker processes HDFS files, and Grafana displays resource usage and logs. Next contrast the object-storage path: an Iceberg client uses Polaris for catalog operations and SeaweedFS for files. Finally show Kafka as the event-streaming path, and identify YARN/Hive/later tools as separate sessions.

The architecture is deliberately small for demonstration. Managed cloud clusters, multiple nodes, redundant data services, external persistent storage, private credentials and backup policies require a different deployment design. For general Kubernetes concepts, continue with the [illustrated introduction](../00-kubernetes-introduction/00-kubernetes-introduction.ipynb); for this repository's exact commands, use the [quick start](QUICKSTART.md) or setup notebooks.